In [1]:
%matplotlib inline
import os, datetime
import netCDF4 as nc
from netCDF4 import Dataset
import cftime
import numpy as np
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import signal
import requests
import os
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy import signal
import soundfile as sf

In [2]:
#EDITED TIFFINYS FUNCTION
def j2000_sec_to_datetime(unix_sec):
    dt0 = datetime.datetime(2000, 1, 1, 12)  # J2000 epoch reference
    unix_sec = np.array(unix_sec)
    return np.array([dt0 + datetime.timedelta(seconds=s) for s in unix_sec])

def datetime_to_day(dt_array):
    # Ensure dt_array is a NumPy array of datetime objects
    dt_array = np.array(dt_array)
    t0 = dt_array[0]
    unix_seconds = np.array([(dt - t0).total_seconds() for dt in dt_array])
    return unix_seconds

def timeTicks(x, pos):
    d = datetime.timedelta(seconds=x)
    hours, remainder = divmod(int(d.total_seconds()), 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours}:{minutes:02}:{seconds:02}"

def format_time_axis(ax):
    locator = mdates.HourLocator(interval=1)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_minor_locator(mdates.HourLocator())
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    return b, a

def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = signal.lfilter(b, a, data)
    return y

from scipy import signal
import numpy as np

def butter_bandpass_filter(data, lowcut, highcut, fs, order=4, pad_length=500):
    # same as tiffiny's
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')

    # create reflection padding
    start_pad = data[pad_length:0:-1]
    end_pad = data[-1:-pad_length-1:-1]
    padded_data = np.concatenate([start_pad, data, end_pad])

    # apply zero-phase filter
    filtered_padded = signal.filtfilt(b, a, padded_data)

    # remove padding so it doesn't impact future data stuff
    filtered_data = filtered_padded[pad_length:-pad_length]

    return filtered_data

In [3]:
# Path to your external drive
external_drive = "/Volumes/lasp_hd"

# New folder name
new_folder = "202201_goes16_fullmonth" #change according to the dates you need

# Full path
full_path = os.path.join(external_drive, new_folder)

# Create the folder
os.makedirs(full_path, exist_ok=True)

# Remote data base URL
dataurl = 'https://data.ngdc.noaa.gov/platforms/solar-space-observing-satellites/goes/goes16/l2/data/magn-l2-hires'

# Set date range
start_date = datetime.datetime(2022, 1, 1) #change this
end_date = datetime.datetime(2022, 1, 31)  # inclusive

# Loop through each day
current_date = start_date
while current_date <= end_date:
    yyyymmdd = current_date.strftime("%Y%m%d")
    year = current_date.strftime("%Y")
    month = current_date.strftime("%m")

    filename = f"dn_magn-l2-hires_g16_d{yyyymmdd}_v1-0-1.nc"
    subdir = f"/{year}/{month}/"
    url_path = dataurl + subdir + filename
    save_path = os.path.join(full_path, filename)

    # Download if missing
    if not os.path.exists(save_path):
        print(f"\nDownloading {filename}...")
        try:
            r = requests.get(url_path, timeout=15)
            if r.status_code == 200:
                with open(save_path, "wb") as f:
                    f.write(r.content)
                print(f" Saved to: {save_path}")
            else:
                print(f" Failed ({r.status_code}) for {filename}")
        except Exception as e:
            print(f" Error downloading {filename}: {e}")
    else:
        print(f" Already exists: {filename}")

    current_date += datetime.timedelta(days=1)

 Already exists: dn_magn-l2-hires_g16_d20220101_v1-0-1.nc
 Already exists: dn_magn-l2-hires_g16_d20220102_v1-0-1.nc
 Already exists: dn_magn-l2-hires_g16_d20220103_v1-0-1.nc
 Already exists: dn_magn-l2-hires_g16_d20220104_v1-0-1.nc
 Already exists: dn_magn-l2-hires_g16_d20220105_v1-0-1.nc
 Already exists: dn_magn-l2-hires_g16_d20220106_v1-0-1.nc
 Already exists: dn_magn-l2-hires_g16_d20220107_v1-0-1.nc
 Already exists: dn_magn-l2-hires_g16_d20220108_v1-0-1.nc
 Already exists: dn_magn-l2-hires_g16_d20220109_v1-0-1.nc
 Already exists: dn_magn-l2-hires_g16_d20220110_v1-0-1.nc
 Already exists: dn_magn-l2-hires_g16_d20220111_v1-0-1.nc
 Already exists: dn_magn-l2-hires_g16_d20220112_v1-0-1.nc
 Already exists: dn_magn-l2-hires_g16_d20220113_v1-0-1.nc
 Already exists: dn_magn-l2-hires_g16_d20220114_v1-0-1.nc
 Already exists: dn_magn-l2-hires_g16_d20220115_v1-0-1.nc
 Already exists: dn_magn-l2-hires_g16_d20220116_v1-0-1.nc
 Already exists: dn_magn-l2-hires_g16_d20220117_v1-0-1.nc
 Already exist

## cleaning the raw data to make sure that there are no weird points, and that we can use throughout the rest of the pipeline

In [5]:
# Define the time range
start_date = datetime.date(2022, 1, 1) #change this
end_date   = datetime.date(2022, 1, 31) #change this
current_date = start_date

# Define your data folder
full_path = "/Volumes/lasp_hd/202201_goes16_fullmonth/" #change this according to your hard drive

# Dictionaries for per-day data
daily_raw_B = {}
raw_B = {}
daily_dts = {}

while current_date <= end_date:
    yyyymmdd = current_date.strftime("%Y%m%d")
    filename = f"dn_magn-l2-hires_g16_d{yyyymmdd}_v1-0-1.nc"
    file_path = os.path.join(full_path, filename)

    print(f"\nloading raw {filename}")

    try:
        g16a = nc.Dataset(file_path)
        b_total = g16a.variables['b_total'][:]
        raw_time = g16a.variables['time'][:]

        dts = j2000_sec_to_datetime(raw_time)
        dt_start = datetime.datetime.combine(current_date, datetime.time.min)
        dt_end = datetime.datetime.combine(current_date, datetime.time.max)

        # Filter times strictly within the day
        mask = (dts >= dt_start) & (dts <= dt_end)
        filtered_b = b_total[mask]
        filtered_dts = dts[mask]
        raw_only = b_total[mask]

        # Save daily data in dicts, keyed by date string WITHOUT concatenation
        daily_raw_B[f"{yyyymmdd}"] = filtered_b  # magnetic field numeric data
        raw_B[f'{yyyymmdd}'] = raw_only
        daily_dts[f"{yyyymmdd}"] = filtered_dts      # datetime timestamps

        g16a.close()
        print(f"done. {len(filtered_dts)} samples from {yyyymmdd}")

    except Exception as e:
        print(f"error loading {filename}: {e}")

    current_date += datetime.timedelta(days=1)


loading raw dn_magn-l2-hires_g16_d20220101_v1-0-1.nc
done. 864000 samples from 20220101

loading raw dn_magn-l2-hires_g16_d20220102_v1-0-1.nc
done. 863997 samples from 20220102

loading raw dn_magn-l2-hires_g16_d20220103_v1-0-1.nc
done. 863999 samples from 20220103

loading raw dn_magn-l2-hires_g16_d20220104_v1-0-1.nc
done. 863999 samples from 20220104

loading raw dn_magn-l2-hires_g16_d20220105_v1-0-1.nc
done. 864000 samples from 20220105

loading raw dn_magn-l2-hires_g16_d20220106_v1-0-1.nc
done. 863997 samples from 20220106

loading raw dn_magn-l2-hires_g16_d20220107_v1-0-1.nc
done. 864000 samples from 20220107

loading raw dn_magn-l2-hires_g16_d20220108_v1-0-1.nc
done. 863999 samples from 20220108

loading raw dn_magn-l2-hires_g16_d20220109_v1-0-1.nc
done. 864000 samples from 20220109

loading raw dn_magn-l2-hires_g16_d20220110_v1-0-1.nc
done. 864000 samples from 20220110

loading raw dn_magn-l2-hires_g16_d20220111_v1-0-1.nc
done. 863998 samples from 20220111

loading raw dn_magn-

In [6]:
# getting rid of huge outlier points
threshold = np.abs(500)  # setting threshold

for key, data in daily_raw_B.items():
    print(f"\nChecking {key}")

    # Check before replacing
    print('Max before:', np.nanmax(data))
    print('Min before:', np.nanmin(data))

    # Apply threshold: replace values above 500 nT with NaN
    mask = np.abs(data) > threshold
    data_with_nans = data.copy()
    data_with_nans[mask] = np.nan

    # Save back to the dictionary
    daily_raw_B[key] = data_with_nans

    # Check after replacing
    nan_count = np.sum(np.isnan(data_with_nans))
    max_val = np.nanmax(data_with_nans)
    min_val = np.nanmin(data_with_nans)

    print('Max after (ignoring NaNs):', max_val)
    print('Min after (ignoring NaNs):', min_val)
    print('NaN count:', nan_count)


Checking 20220101
Max before: 104.61812
Min before: 50.767845
Max after (ignoring NaNs): 104.61812
Min after (ignoring NaNs): 50.767845
NaN count: 0

Checking 20220102
Max before: 108.2752
Min before: 60.55672
Max after (ignoring NaNs): 108.2752
Min after (ignoring NaNs): 60.55672
NaN count: 0

Checking 20220103
Max before: 108.65241
Min before: 49.170536
Max after (ignoring NaNs): 108.65241
Min after (ignoring NaNs): 49.170536
NaN count: 0

Checking 20220104
Max before: 103.0037
Min before: 67.52457
Max after (ignoring NaNs): 103.0037
Min after (ignoring NaNs): 67.52457
NaN count: 0

Checking 20220105
Max before: 107.81003
Min before: 76.72531
Max after (ignoring NaNs): 107.81003
Min after (ignoring NaNs): 76.72531
NaN count: 0

Checking 20220106
Max before: 109.89357
Min before: 89.23207
Max after (ignoring NaNs): 109.89357
Min after (ignoring NaNs): 89.23207
NaN count: 0

Checking 20220107
Max before: 110.04553
Min before: 86.3869
Max after (ignoring NaNs): 110.04553
Min after (ign

In [7]:
from scipy.interpolate import interp1d

def linear_interpolate(data, time):
    time_sec = np.array([t.timestamp() for t in time])
    data = np.array(data)

    valid = ~np.isnan(data)
    if np.sum(valid) < 2:
        return data  # Not enough points to interpolate

    mean_value = np.nanmean(data)

    # Main linear interpolation
    linear_interp = interp1d(
        time_sec[valid],
        data[valid],
        kind='linear',
        bounds_error=False,
        fill_value=np.nan)

    interp_data = linear_interp(time_sec)

    # Find NaN indices (likely only at edges)
    isnan = np.isnan(interp_data)

    # Fill NaNs (edges or others that couldn't interpolate) with median
    interp_data[isnan] = mean_value

    return interp_data

In [8]:
#interpolating across gaps before the despike
for key in daily_raw_B:
    daily_raw_B[key] = linear_interpolate(daily_raw_B[key], daily_dts[key])

## now despiking the data

In [10]:
import utils
from utils import find_neighbors

#adapting despike to this data
def despike(data, time, dBdt_th = np.abs(5),num=10): #can change threshold if needed, the other input is the data
    Bdt = [(t2 - t1).total_seconds() for t1,t2 in zip(time[:-1], time[1:])]    
    dBxdt = np.diff(data)/Bdt
     
    spike_flag = np.abs(dBxdt) > dBdt_th
    flag = np.append(spike_flag,[False])
    flag_nb = utils.find_neighbors(flag,num=num)

    despike_time = time 
    despike_B = data.copy()
    despike_B[flag_nb] = np.nan

    return despike_B, despike_time

In [11]:
for key in daily_raw_B:
    data = daily_raw_B[key]
    time = daily_dts[key]

    despike_B, despike_time = despike(data, time, dBdt_th = np.abs(5), num = 10)

    daily_raw_B[key] = despike_B
    daily_dts[key] = despike_time

In [12]:
def check_nans_in_daily_data(data_dict):
    for key, data in data_dict.items():
        nan_count = np.sum(np.isnan(data))
        total = data.size

        print(f"{key}: NaN count = {nan_count}")
check_nans_in_daily_data(daily_raw_B)

20220101: NaN count = 0
20220102: NaN count = 0
20220103: NaN count = 288
20220104: NaN count = 103
20220105: NaN count = 42
20220106: NaN count = 0
20220107: NaN count = 62
20220108: NaN count = 909
20220109: NaN count = 20
20220110: NaN count = 25
20220111: NaN count = 0
20220112: NaN count = 65
20220113: NaN count = 0
20220114: NaN count = 1236
20220115: NaN count = 623
20220116: NaN count = 684
20220117: NaN count = 42
20220118: NaN count = 29
20220119: NaN count = 3031
20220120: NaN count = 0
20220121: NaN count = 62
20220122: NaN count = 422
20220123: NaN count = 0
20220124: NaN count = 42
20220125: NaN count = 331
20220126: NaN count = 104
20220127: NaN count = 0
20220128: NaN count = 42
20220129: NaN count = 956
20220130: NaN count = 164
20220131: NaN count = 108


In [13]:
#interpolating across these huge nan gaps
for key in daily_raw_B:
    daily_raw_B[key] = linear_interpolate(daily_raw_B[key], daily_dts[key])

check_nans_in_daily_data(daily_raw_B)

20220101: NaN count = 0
20220102: NaN count = 0
20220103: NaN count = 0
20220104: NaN count = 0
20220105: NaN count = 0
20220106: NaN count = 0
20220107: NaN count = 0
20220108: NaN count = 0
20220109: NaN count = 0
20220110: NaN count = 0
20220111: NaN count = 0
20220112: NaN count = 0
20220113: NaN count = 0
20220114: NaN count = 0
20220115: NaN count = 0
20220116: NaN count = 0
20220117: NaN count = 0
20220118: NaN count = 0
20220119: NaN count = 0
20220120: NaN count = 0
20220121: NaN count = 0
20220122: NaN count = 0
20220123: NaN count = 0
20220124: NaN count = 0
20220125: NaN count = 0
20220126: NaN count = 0
20220127: NaN count = 0
20220128: NaN count = 0
20220129: NaN count = 0
20220130: NaN count = 0
20220131: NaN count = 0


## now we are ready to do the butterworth band pass filter

In [15]:
#from Tiffiny's code: filter parameters
fs = 10  # Sampling frequency in Hz
lowcut = 0.1  # Lower bound of bandpass (Hz)
highcut = 4.0  # Upper bound of bandpass (Hz)
N = 1  # Filter order
NFFT = 1024  # FFT segment length
freqlim = 1  # Frequency limit for plotting
db_min, db_max = -20, 20  # Power level range for plotting
str_plt, end_plt = 0, 1066  # Indexes for plotting

# this is for each individual day 
daily_filtered_B = {}

for day_key in daily_raw_B:
    filtered_B = butter_bandpass_filter(daily_raw_B[day_key], lowcut, highcut, fs, order=4)
    daily_filtered_B[day_key] = filtered_B

In [16]:
for key in daily_filtered_B:
    data = daily_filtered_B[key]
    time = daily_dts[key]

    despike_filtered_B, despike_time = despike(data, time, dBdt_th = np.abs(5), num = 10)

    daily_filtered_B[key] = despike_filtered_B
    daily_dts[key] = despike_time

for key, data in daily_filtered_B.items():
    print(f"\nChecking {key}")

    # Check before replacing
    print('Max before:', np.nanmax(data))
    print('Min before:', np.nanmin(data))
    
    # Apply threshold: replace values above 7 nT with NaN
    mask = np.abs(data) > threshold
    filtered_data_with_nans = data.copy()
    filtered_data_with_nans[mask] = np.nan

    # Save back to the dictionary
    daily_filtered_B[key] = filtered_data_with_nans

    # Check after replacing
    nan_count = np.sum(np.isnan(filtered_data_with_nans))
    max_val = np.nanmax(filtered_data_with_nans)
    min_val = np.nanmin(filtered_data_with_nans)

    print('Max after (ignoring NaNs):', max_val)
    print('Min after (ignoring NaNs):', min_val)
    print('NaN count:', nan_count)


Checking 20220101
Max before: 1.0343882974587977
Min before: -0.9854894649494668
Max after (ignoring NaNs): 1.0343882974587977
Min after (ignoring NaNs): -0.9854894649494668
NaN count: 0

Checking 20220102
Max before: 0.44752101887049983
Min before: -0.49867749503835834
Max after (ignoring NaNs): 0.44752101887049983
Min after (ignoring NaNs): -0.49867749503835834
NaN count: 0

Checking 20220103
Max before: 1.932778714359402
Min before: -2.406721629036878
Max after (ignoring NaNs): 1.932778714359402
Min after (ignoring NaNs): -2.406721629036878
NaN count: 0

Checking 20220104
Max before: 0.8749140755937246
Min before: -0.7547544602359071
Max after (ignoring NaNs): 0.8749140755937246
Min after (ignoring NaNs): -0.7547544602359071
NaN count: 0

Checking 20220105
Max before: 1.2310427286996641
Min before: -1.2533801741647654
Max after (ignoring NaNs): 1.2310427286996641
Min after (ignoring NaNs): -1.2533801741647654
NaN count: 0

Checking 20220106
Max before: 0.4836272303655925
Min before

In [17]:
#interpolating across these huge nan gaps
for key in daily_filtered_B:
    daily_filtered_B[key] = linear_interpolate(daily_filtered_B[key], daily_dts[key])

check_nans_in_daily_data(daily_filtered_B)

20220101: NaN count = 0
20220102: NaN count = 0
20220103: NaN count = 0
20220104: NaN count = 0
20220105: NaN count = 0
20220106: NaN count = 0
20220107: NaN count = 0
20220108: NaN count = 0
20220109: NaN count = 0
20220110: NaN count = 0
20220111: NaN count = 0
20220112: NaN count = 0
20220113: NaN count = 0
20220114: NaN count = 0
20220115: NaN count = 0
20220116: NaN count = 0
20220117: NaN count = 0
20220118: NaN count = 0
20220119: NaN count = 0
20220120: NaN count = 0
20220121: NaN count = 0
20220122: NaN count = 0
20220123: NaN count = 0
20220124: NaN count = 0
20220125: NaN count = 0
20220126: NaN count = 0
20220127: NaN count = 0
20220128: NaN count = 0
20220129: NaN count = 0
20220130: NaN count = 0
20220131: NaN count = 0


In [18]:
for key in daily_filtered_B:
    data = daily_filtered_B[key]
    print('mean value:', np.mean(daily_filtered_B[key]))

mean value: 3.7746163815772444e-08
mean value: 2.9211901844131325e-08
mean value: -1.584958743509095e-08
mean value: -1.3302684965144946e-08
mean value: 9.618196564970021e-08
mean value: 1.0524662733923317e-08
mean value: 5.518584265200132e-08
mean value: 2.7720867342845875e-06
mean value: -1.8624966589443537e-07
mean value: 4.7431121671994666e-08
mean value: -4.107025643899491e-08
mean value: 1.9430613521427447e-08
mean value: 1.0455695809357477e-08
mean value: -3.8246688050840054e-08
mean value: 4.294203051285491e-08
mean value: 2.361142885912092e-06
mean value: -5.107837775112757e-08
mean value: 6.445691082277025e-08
mean value: 1.3954136058081848e-05
mean value: -1.5416019527857562e-08
mean value: -1.7194110752145193e-09
mean value: 1.1565815623584079e-07
mean value: -4.8633951739655055e-08
mean value: 5.355802587160742e-08
mean value: -4.936141196548997e-08
mean value: -7.879383140186241e-08
mean value: -4.6695113496071227e-08
mean value: 3.00304860841573e-08
mean value: 6.9842286

## normalization and paul stretch for .wav files

In [20]:
import copy
import pandas as pd

import matplotlib.colors as colors

import soundfile as sf
from paulstretch_mono1 import paulstretch

In [21]:
def thm_fgm_paulstretch(times, data, stretch=1, window=512./1024, samplerate=1024, return_time=False):
    # Window for paulstretch is specified so as to be equivalent to a window of 512 samples 
    # when using the default sample rate of 44100
    paulStretch_data = paulstretch(data, stretch, window, samplerate=samplerate)
    
    if return_time == False:
        return paulStretch_data
    else:
        epochs = [ii.timestamp() for ii in times]
        epoch_stretch = np.linspace(epochs[0], epochs[-1], int(len(times) * stretch))
        epoch_stretch = epoch_stretch[:len(paulStretch_data)]
        times_interp_dt = np.array([datetime.datetime.fromtimestamp(ii) for ii in epoch_stretch])
        return times_interp_dt, paulStretch_data

In [22]:
#paulstretch parameters
window = 512. / 44100  # ~0.0116 seconds
stretch = 1            # stretch factor
samplerate = 44100

#apply to daily files 
daily_paulstretched = {}

for day_key in daily_filtered_B:
    times = daily_dts[day_key.replace("raw", "dts")]
    data = daily_filtered_B[day_key]

    paul_times, paul_data = thm_fgm_paulstretch(
        times, data,
        samplerate=samplerate,
        stretch=stretch,
        window=window,
        return_time=True)

    daily_paulstretched[day_key] = {
        "times": paul_times,
        "data": paul_data}

In [23]:
# finding the maximum for each day individually so that it can be used for the global max
max_values = []

for key in daily_filtered_B:
    data = daily_filtered_B[key]
    max_val = np.max(data) 
    max_values.append(max_val)

# convert list to array so that we can do math and apply functions to it later
max_values_array = np.array(max_values)

print("Max values array:", max_values_array)

Max values array: [1.0343883  0.44752102 1.93277871 0.87491408 1.23104273 0.48362723
 1.30996316 2.0014057  1.08767259 0.68661067 0.49322789 1.09893256
 0.50023537 1.63416187 0.82506793 2.27224474 1.06669382 0.59442275
 2.37261013 0.45404766 1.33491396 2.268701   0.81277152 1.22492295
 2.10113263 2.05656076 0.39230576 1.0142933  2.09974531 2.1762488
 1.52852186]


In [24]:
# global max from full week of paulstretched data
global_max = np.max(max_values_array)

# normalizing each day by same max so they all have consistent volume
daily_paulstretched_norm = {}

for day_key, day_data in daily_paulstretched.items():
    times = day_data["times"]
    data = day_data["data"]
    normalized_data = data / global_max
    daily_paulstretched_norm[day_key] = {
        "times": times,
        "data": normalized_data}

## looking good! now time to make the audio (.wav) files

In [26]:
# daily_paulstretched is a dict with keys like 'raw20220209' containing {'times': ..., 'data': ...}
for day_key, day_data in daily_paulstretched.items():
    times = day_data['times']
    data = day_data['data']
    
    # Normalize by the global max from the full week
    data_norm = data / global_max
    
    yyyymmdd = day_key.replace('raw', '')
    filename = f"{yyyymmdd}_paulstretch.wav"
    
    # Save normalized daily audio as 32-bit float WAV
    sf.write(os.path.join(full_path, filename), data_norm.astype(np.float32), samplerate)
    print(f"saved daily normalized .wav: {filename}")

saved daily normalized .wav: 20220101_paulstretch.wav
saved daily normalized .wav: 20220102_paulstretch.wav
saved daily normalized .wav: 20220103_paulstretch.wav
saved daily normalized .wav: 20220104_paulstretch.wav
saved daily normalized .wav: 20220105_paulstretch.wav
saved daily normalized .wav: 20220106_paulstretch.wav
saved daily normalized .wav: 20220107_paulstretch.wav
saved daily normalized .wav: 20220108_paulstretch.wav
saved daily normalized .wav: 20220109_paulstretch.wav
saved daily normalized .wav: 20220110_paulstretch.wav
saved daily normalized .wav: 20220111_paulstretch.wav
saved daily normalized .wav: 20220112_paulstretch.wav
saved daily normalized .wav: 20220113_paulstretch.wav
saved daily normalized .wav: 20220114_paulstretch.wav
saved daily normalized .wav: 20220115_paulstretch.wav
saved daily normalized .wav: 20220116_paulstretch.wav
saved daily normalized .wav: 20220117_paulstretch.wav
saved daily normalized .wav: 20220118_paulstretch.wav
saved daily normalized .wav:

In [60]:
new_folder = "zooniverse_01"
fuller_path = os.path.join(full_path, new_folder)
os.makedirs(fuller_path, exist_ok=True)
# --- 1. Spectrogram Configuration Parameters ---
NFFT, noverlap, db_min, db_max, freqlim, fs = 1024, 1024 // 4, -35, 5, 2.3, 10

# --- 2. Iteration Loop ---
for day_key, day_data in daily_paulstretched.items():
    times = day_data['times']
    data = day_data['data']
    
    yyyymmdd = day_key.replace('raw', '')
    current_day = datetime.datetime.strptime(yyyymmdd, "%Y%m%d")
    raw_day_data = daily_raw_B.get(day_key, data)
    
    # 💡 FIX 1: Tightly bound figure, but with vertical padding for title/colorbar
    fig, ax_spec = plt.subplots(figsize=(12, 6))
    fig.subplots_adjust(left=0, right=1, bottom=0.08, top=0.92) # Stretches left/right to 0 and 1
    
    if data is not None and len(data) > 0:
        f, t, Sxx = signal.spectrogram(data, fs=fs, nperseg=NFFT, noverlap=noverlap, detrend="linear")
        abs_times_utc = pd.to_datetime([current_day + datetime.timedelta(seconds=float(sec)) for sec in t]).tz_localize("UTC")
        
        mesh = ax_spec.pcolormesh(abs_times_utc, f, 10 * np.log10(Sxx + 1e-12), cmap="plasma", shading="auto", vmin=db_min, vmax=db_max)
        
        raw_ds = np.array([np.mean(chunk) for chunk in np.array_split(raw_day_data, len(t))])
        
        ax_spec.plot(abs_times_utc, 0.015221 * raw_ds, label="$H^+$", color="magenta", linewidth=1.4)
        ax_spec.plot(abs_times_utc, 0.003805 * raw_ds, label="$He^+$", color="cyan", linewidth=1.4)
        ax_spec.plot(abs_times_utc, 0.0009513 * raw_ds, label="$O^+$", color="lightcyan", linewidth=1.4)
        
        # 💡 FIX 2: Dynamic bounds & axis line containment
        ax_spec.set_xlim(abs_times_utc.min(), abs_times_utc.max())
        ax_spec.set_yscale("log")
        ax_spec.set_ylim(0.075, freqlim)
        
        # 💡 FIX 3: Push Y-Axis numbers AND Title inside the plot matrix
        ax_spec.tick_params(axis='y', direction='in', colors='white', pad=-25, labelsize=12, which='both')
        ax_spec.text(0.01, 0.5, "Frequency (Hz)", color="white", fontsize=14, rotation=90, 
                     va="center", ha="left", transform=ax_spec.transAxes, fontweight="bold")
        
        # 💡 FIX 4: Normal X-Axis but styled to sit safely at the bottom
        ax_spec.tick_params(axis='x', direction='in', colors='white', pad=-15, labelsize=11)
        
        # 💡 FIX 5: Put Title and Legend INSIDE the visualization space
        ax_spec.set_title(f"Spectrogram for {current_day.strftime('%B %d, %Y')}", 
                          color="white", x=0.01, y=0.93, loc="left", fontsize=13, fontweight="bold")
        ax_spec.legend(loc="upper right", bbox_to_anchor=(0.99, 0.98), fontsize=11, framealpha=0.4, facecolor='black', edgecolor='none', labelcolor='white')
        
        ax_spec.grid(False)
        
    else:
        ax_spec.text(0.5, 0.5, "No Spectrogram Data Found", ha="center", va="center", color="white", fontsize=12)
        ax_spec.set_facecolor('black')

    # Save figure with precise zero margins on the sides
    spec_filename = f"{yyyymmdd}_spectrogram.png"
    plt.savefig(
        os.path.join(fuller_path, spec_filename), 
        bbox_inches=None, 
        pad_inches=0, 
        dpi=150
    )
    plt.close(fig)
    print(f"Generated clean-edge visual with internal text: {spec_filename}")

Generated clean-edge visual with internal text: 20220101_spectrogram.png
Generated clean-edge visual with internal text: 20220102_spectrogram.png
Generated clean-edge visual with internal text: 20220103_spectrogram.png
Generated clean-edge visual with internal text: 20220104_spectrogram.png
Generated clean-edge visual with internal text: 20220105_spectrogram.png
Generated clean-edge visual with internal text: 20220106_spectrogram.png
Generated clean-edge visual with internal text: 20220107_spectrogram.png
Generated clean-edge visual with internal text: 20220108_spectrogram.png
Generated clean-edge visual with internal text: 20220109_spectrogram.png
Generated clean-edge visual with internal text: 20220110_spectrogram.png
Generated clean-edge visual with internal text: 20220111_spectrogram.png
Generated clean-edge visual with internal text: 20220112_spectrogram.png
Generated clean-edge visual with internal text: 20220113_spectrogram.png
Generated clean-edge visual with internal text: 202

In [62]:
import os
import soundfile as sf
import numpy as np

# daily_paulstretched is a dict with keys like 'raw20220209' containing {'times': ..., 'data': ...}
for day_key, day_data in daily_paulstretched.items():
    times = day_data['times']
    data = day_data['data']
    
    # Normalize by the global max from the full week
    data_norm = data / global_max
    yyyymmdd = day_key.replace('raw', '')
    
    # 1. Force the numpy array data to Mono if it isn't already
    # If data is 2D (stereo), average the channels together
    if len(data_norm.shape) > 1 and data_norm.shape[1] > 1:
        data_mono = np.mean(data_norm, axis=1)
    else:
        data_mono = data_norm

    # 2. Use .ogg extension (Zooniverse compatible and highly compressed)
    ogg_filename = f"{yyyymmdd}_paulstretch.mp3"
    ogg_path = os.path.join(fuller_path, ogg_filename)
    
    # 3. Save directly as a compressed OGG file (Vorbis format)
    # This keeps files incredibly small without needing ffmpeg
    sf.write(ogg_path, data_mono.astype(np.float32), samplerate, format='ogg', subtype='VORBIS')
    
    print(f"saved daily compressed Zooniverse .mp3: {ogg_filename}")

saved daily compressed Zooniverse .mp3: 20220101_paulstretch.mp3
saved daily compressed Zooniverse .mp3: 20220102_paulstretch.mp3
saved daily compressed Zooniverse .mp3: 20220103_paulstretch.mp3
saved daily compressed Zooniverse .mp3: 20220104_paulstretch.mp3
saved daily compressed Zooniverse .mp3: 20220105_paulstretch.mp3
saved daily compressed Zooniverse .mp3: 20220106_paulstretch.mp3
saved daily compressed Zooniverse .mp3: 20220107_paulstretch.mp3
saved daily compressed Zooniverse .mp3: 20220108_paulstretch.mp3
saved daily compressed Zooniverse .mp3: 20220109_paulstretch.mp3
saved daily compressed Zooniverse .mp3: 20220110_paulstretch.mp3
saved daily compressed Zooniverse .mp3: 20220111_paulstretch.mp3
saved daily compressed Zooniverse .mp3: 20220112_paulstretch.mp3
saved daily compressed Zooniverse .mp3: 20220113_paulstretch.mp3
saved daily compressed Zooniverse .mp3: 20220114_paulstretch.mp3
saved daily compressed Zooniverse .mp3: 20220115_paulstretch.mp3
saved daily compressed Zo

In [64]:
import os
import csv

# 1. RUN THE LOOP
manifest_data = []

# This fits right into your existing daily processing loop
for day_key, day_data in daily_paulstretched.items():
    yyyymmdd = day_key.replace('raw', '')
    
    # Define your exact local filenames
    mp3_filename = f"{yyyymmdd}_paulstretch.mp3"
    png_filename = f"{yyyymmdd}_spectrogram.png"  
    
    # Append the raw local filenames directly to the manifest data list
    manifest_data.append({
        'png_file': png_filename,
        'mp3_file': mp3_filename,
        'day_id': yyyymmdd
    })

# 2. WRITE THE MANIFEST TO A CSV FILE
# Saves 'january_manifest.csv' to the 'full_path' folder on your machine
csv_path = os.path.join(fuller_path, "january_manifest.csv")

with open(csv_path, mode='w', newline='') as f:
    # Column 1 = PNG, Column 2 = MP3 to perfectly match your terminal command
    writer = csv.DictWriter(f, fieldnames=['png_file', 'mp3_file', 'day_id'])
    writer.writeheader()
    writer.writerows(manifest_data)

print(f"Successfully created local filename CSV manifest at: {csv_path}")

Successfully created local filename CSV manifest at: /Volumes/lasp_hd/202201_goes16_fullmonth/zooniverse_01/january_manifest.csv
